# M7: Streaming + Abort

In M6 the client saw nothing until the whole answer was done, and a client that left kept its request running to the end. M7 fixes both.

**Streaming.** `stream: true` on `/generate` returns Server-Sent Events: one `data: {...}` chunk per scheduler step, then `data: [DONE]`. Each `Req` now has an `out_queue`. The scheduler puts one item on it per step, and the handler turns each item into a chunk.

```text
scheduler thread                               handler (stream_results)
process_batch_result ──put((tok, reason))──▶  out_queue ──get()──▶ decode ──▶ data: {...}
abort_request        ──put((None, ABORT))──▶             ──get()──▶ last chunk ──▶ data: [DONE]
```

| Item | When |
|---|---|
| `(tok, None)` | a new token, not finished |
| `(tok, FINISH_LENGTH / FINISH_MATCHED_TOKEN)` | the last token |
| `(None, FINISH_ERROR / FINISH_ABORT)` | ended without a new token |

**Abort.** When the client disconnects, the handler sends an `AbortReq(rid)` to the scheduler's inbox. At its next step the scheduler removes the request, waiting or running, and it ends as `ABORTED` with `FINISH_ABORT`. Abort is one more edge in the M6 state machine:

```text
WAITING ──▶ RUNNING ──▶ FINISHED
   │           ├──────▶ FAILED
   │           └──────▶ ABORTED
   └──────────────────▶ ABORTED
```

## Setup

Same as M6: start uvicorn in a background thread so the notebook can act as the client.

Port 30000 is SGLang's default. Stop any `python server.py` you started in a terminal first, or the port is taken.

In [1]:
import json
import threading
import time

import httpx
import uvicorn

from detokenizer import REPLACEMENT_CHAR, IncrementalDetokenizer
from engine import Engine
from req import _TRANSITIONS, AbortReq, ReqStatus
from server import create_app

engine = Engine()
runner = engine.model_runner
tok = engine.tokenizer
print(runner.device)

server = uvicorn.Server(uvicorn.Config(create_app(engine), host="127.0.0.1", port=30000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

client = httpx.Client(base_url="http://127.0.0.1:30000", timeout=300)


def show(text):
    """repr() with U+FFFD spelled out, so it reads the same in any font."""
    return repr(text).replace(REPLACEMENT_CHAR, "<U+FFFD>")


def sse_events(resp):
    """Yield the data payload of each SSE message."""
    for line in resp.iter_lines():
        if line.startswith("data: "):
            yield line.removeprefix("data: ")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

mps


## What a Stream Looks Like

Every chunk has the same fields as the non-stream response. `text` is everything decoded so far, not just the new piece, as in SGLang's default. `finish_reason` is `null` until the last chunk. Then `[DONE]` closes the stream.

In [2]:
body = {"text": "Write a short story about a dragon", "sampling_params": {"max_new_tokens":20}, "stream": True}
with client.stream("POST", "/generate", json=body) as resp:
    print(resp.status_code, resp.headers["content-type"])
    for data in sse_events(resp):
        if data == "[DONE]":
            print("[DONE]")
            break
        chunk = json.loads(data)
        print(f"{chunk['meta_info']['completion_tokens']:>2} tok  {chunk['text']!r:<45} finish_reason={chunk['meta_info']['finish_reason']}")

200 text/event-stream; charset=utf-8


 1 tok  ' who'                                        finish_reason=None
 2 tok  ' who is'                                     finish_reason=None


 3 tok  ' who is a'                                   finish_reason=None
 4 tok  ' who is a master'                            finish_reason=None
 5 tok  ' who is a master of'                         finish_reason=None


 6 tok  ' who is a master of the'                     finish_reason=None
 7 tok  ' who is a master of the art'                 finish_reason=None


 8 tok  ' who is a master of the art of'              finish_reason=None
 9 tok  ' who is a master of the art of illusion'     finish_reason=None
10 tok  ' who is a master of the art of illusion,'    finish_reason=None


11 tok  ' who is a master of the art of illusion, and' finish_reason=None
12 tok  ' who is a master of the art of illusion, and describe' finish_reason=None


13 tok  ' who is a master of the art of illusion, and describe the' finish_reason=None
14 tok  ' who is a master of the art of illusion, and describe the setting' finish_reason=None


15 tok  ' who is a master of the art of illusion, and describe the setting,' finish_reason=None
16 tok  ' who is a master of the art of illusion, and describe the setting, characters' finish_reason=None


17 tok  ' who is a master of the art of illusion, and describe the setting, characters,' finish_reason=None
18 tok  ' who is a master of the art of illusion, and describe the setting, characters, and' finish_reason=None


19 tok  ' who is a master of the art of illusion, and describe the setting, characters, and plot' finish_reason=None
20 tok  ' who is a master of the art of illusion, and describe the setting, characters, and plot.' finish_reason={'type': 'length', 'length': 20}
[DONE]


## Streaming Lowers TTFT, Not Total Latency

Streaming changes *when* output reaches the client, not how fast it is generated. Time to first token (TTFT) drops to about one step. The total time stays about the same, because the scheduler does exactly the same work.

In [3]:
body = {"text": "Write a short story about a dragon.", "sampling_params": {"max_new_tokens": 48}}
client.post("/generate", json=body)  # warm up, so the first timed run does not pay for it

start = time.perf_counter()
plain = client.post("/generate", json=body).json()
plain_total = time.perf_counter() - start

start = time.perf_counter()
first = None
with client.stream("POST", "/generate", json={**body, "stream": True}) as resp:
    for data in sse_events(resp):
        if first is None:
            first = time.perf_counter() - start
        if data == "[DONE]":
            break
        last = json.loads(data)
stream_total = time.perf_counter() - start

print(f"non-stream: first byte {plain_total:5.2f}s  total {plain_total:5.2f}s")
print(f"stream    : first byte {first:5.2f}s  total {stream_total:5.2f}s")
print("same tokens:", last["output_ids"] == plain["output_ids"])

non-stream: first byte  3.19s  total  3.19s
stream    : first byte  0.03s  total  3.22s
same tokens: True


## Tokens Are Not Characters

Qwen3 uses byte-level BPE. A token is a run of bytes, and a rare character can be split across tokens. `龘` is 3 UTF-8 bytes, and here it takes two tokens. Decoding only the first gives U+FFFD, the replacement character, shown below as `<U+FFFD>`.

So the handler cannot decode each token on its own and append the text.

In [4]:
ids = tok.encode("好龘好")
print("ids:", ids)
for tid in ids:
    print(f"  {tid:>6} -> bytes {tok.convert_ids_to_tokens(tid)!r:<10} decode alone {show(tok.decode([tid]))}")
print("token by token:", show("".join(tok.decode([t]) for t in ids)))
print("all at once   :", show(tok.decode(ids)))

ids: [52801, 82912, 246, 52801]
   52801 -> bytes 'å¥½'      decode alone '好'
   82912 -> bytes 'é¾'       decode alone '<U+FFFD>'
     246 -> bytes 'ĺ'        decode alone '<U+FFFD>'
   52801 -> bytes 'å¥½'      decode alone '好'
token by token: '好<U+FFFD><U+FFFD>好'
all at once   : '好龘好'


## Incremental Detokenization

Decoding all of `output_ids` again on every step would work, but it costs O(n) per step. `IncrementalDetokenizer` decodes only a small window, with two offsets named after SGLang's `DecodeStatus`:

- `read_offset`: `output_ids[:read_offset]` are already in `decoded_text`.
- `surr_offset`: where the window starts. The tokens from `surr_offset` to `read_offset` are decoded again as context, and only the text beyond them is new.

If the new text ends in U+FFFD, the detokenizer holds it back and leaves the offsets alone, so the next token decodes it again. `finished=True` returns held text anyway, so the last chunk matches the non-stream `text`.

In [5]:
detok = IncrementalDetokenizer(tok)
for i in range(1, len(ids) + 1):
    text = detok.decode(ids[:i], finished=False)
    print(f"step {i}: surr={detok.surr_offset} read={detok.read_offset}  text={show(text)}")

cut = tok.encode("好龘")[:-1]  # stopped in the middle of 龘
print("\nfinished mid-character:", show(IncrementalDetokenizer(tok).decode(cut, finished=True)))

step 1: surr=0 read=1  text='好'
step 2: surr=0 read=1  text='好'
step 3: surr=1 read=3  text='好龘'
step 4: surr=3 read=4  text='好龘好'

finished mid-character: '好<U+FFFD>'


## Inside `out_queue`

Skip HTTP and read a `Req`'s `out_queue` directly. `Engine.submit` returns right away; `Engine.generate` is `submit` plus `done.wait()`.

Only the last item has a finish reason. The reason has to travel with the token: the handler runs behind the scheduler, so by the time it reads token 2, `req.finished_reason` may already be set for a later step.

In [6]:
req = engine.submit("The capital of France is", max_new_tokens=5)
req.done.wait()
while not req.out_queue.empty():
    token_id, reason = req.out_queue.get()
    print(f"({token_id}, {reason.to_json() if reason else None})  {tok.decode([token_id])!r}")

(12095, None)  ' Paris'
(13, None)  '.'
(576, None)  ' The'
(6722, None)  ' capital'
(315, {'type': 'length', 'length': 5})  ' of'


## Abort a Waiting and a Running Request

Submit two long requests. Capacity is still one, so B waits behind A. Abort B while it waits, then abort A after a few tokens.

`abort_request` handles both cases the same way: find the request, set `FINISH_ABORT`, put `(None, reason)` so a stream hears that it ended, then `end_req`. B never ran, so it has no tokens. A stops at the next step boundary: a forward already running cannot be interrupted.

In [7]:
print("transitions into ABORTED:", sorted(s.name for s, dsts in _TRANSITIONS.items() if ReqStatus.ABORTED in dsts))

a = engine.submit("A: Write a long story about a dragon.", max_new_tokens=200)
b = engine.submit("B: Write a long poem about the sea.", max_new_tokens=200)
while len(a.output_ids) < 3:
    time.sleep(0.01)
print(f"before: A={a.status.name} ({len(a.output_ids)} tok)  B={b.status.name}")

engine.abort(b)
b.done.wait()
print(f"abort B: B={b.status.name} {b.finished_reason.to_json()} tokens={len(b.output_ids)}  A={a.status.name}")

engine.abort(a)
a.done.wait()
print(f"abort A: A={a.status.name} tokens={len(a.output_ids)}  last item={a.out_queue.queue[-1][0]}, {a.out_queue.queue[-1][1].to_json()}")

transitions into ABORTED: ['RUNNING', 'WAITING']


before: A=RUNNING (3 tok)  B=WAITING
abort B: B=ABORTED {'type': 'abort', 'message': 'Aborted'} tokens=0  A=RUNNING


abort A: A=ABORTED tokens=5  last item=None, {'type': 'abort', 'message': 'Aborted'}


Aborting a request that already ended does nothing. The scheduler finds no request with that `rid`, and `_TRANSITIONS` has no edge out of a terminal state anyway. This matters because the stream handler sends an abort after *every* stream, including ones that finished normally.

In [8]:
done_req = engine.generate("Hello", max_new_tokens=2)
engine.abort(done_req)
engine.scheduler.submit(AbortReq("no-such-rid"))
print(engine.generate("Hi", max_new_tokens=2).status.name, "| earlier request still:", done_req.status.name)

FINISHED | earlier request still: FINISHED


## Abort on Client Disconnect

Over HTTP, abort comes from `StreamingResponse(background=BackgroundTask(engine.abort, req))`. Starlette runs the background task when the stream ends, and also when the client disconnects.

Wrap `engine.submit` to keep the `Req` the handler creates. Read three chunks, then leave the `with` block, which closes the connection.

In [9]:
submitted = []
real_submit = engine.submit


def recording_submit(*args, **kwargs):
    submitted.append(real_submit(*args, **kwargs))
    return submitted[-1]


engine.submit = recording_submit

body = {"text": "Write a long story about a dragon.", "sampling_params": {"max_new_tokens": 500}, "stream": True}
with client.stream("POST", "/generate", json=body) as resp:
    events = sse_events(resp)
    for _ in range(3):
        print(json.loads(next(events))["text"][-40:])

(req,) = submitted
req.done.wait(timeout=30)
print(f"\nafter disconnect: {req.status.name} {req.finished_reason.to_json()} after {len(req.output_ids)} of 500 tokens")
del engine.submit  # back to the real method

 The
 The story


 The story should

after disconnect: ABORTED {'type': 'abort', 'message': 'Aborted'} after 4 of 500 tokens


## A Failing Step in a Stream

In M6 a failed request became HTTP `500`. A stream cannot do that: the `200` header went out with the first chunk. So the error travels as one more chunk, as in SGLang, followed by `[DONE]`.

In [10]:
def failing_forward(req):
    raise RuntimeError("model crashed")


runner.forward = failing_forward
with client.stream("POST", "/generate", json={"text": "Hello", "stream": True}) as resp:
    print(resp.status_code)
    for data in sse_events(resp):
        print(data)
del runner.forward  # back to the real model

200
{"error": {"message": "model crashed"}}
[DONE]


**Something to think about.** Only the stream path notices a disconnect. The non-stream handler blocks in `done.wait()` and never hears the client leave, so its request runs to the end. SGLang handles this by checking `request.is_disconnected()` every few seconds while it waits.

## Summary

| | M6 | M7 |
|---|---|---|
| Output to the client | one JSON body at the end | `stream: true`: one SSE chunk per step, then `[DONE]` |
| Scheduler to handler | `done` event only | `out_queue`: `(token, finish_reason)` per step, plus `done` |
| Text | `decode(output_ids)` once | `IncrementalDetokenizer`: a small window per step, partial characters held back |
| Client leaves | request runs to the end | `AbortReq` through the inbox; removed at the next step |
| States | `WAITING` / `RUNNING` / `FINISHED` / `FAILED` | plus `ABORTED`, reachable from `WAITING` and `RUNNING` |
| Ending a request | written out in `process_batch_result` | every path calls `end_req` |
| Error in a stream | `500` | `200`, then `data: {"error": ...}` and `[DONE]` |

Streaming lowers TTFT, not end-to-end latency. Abort takes effect only between steps, which is why the scheduler loop works one token at a time.

**Compare with SGLang**: `generate_request()` in `entrypoints/http_server.py`, `stream_output()` in `managers/scheduler_components/output_streamer.py`, `DecodeStatus` in `managers/detokenizer_manager.py`, `abort_request()` in `managers/scheduler.py`, and `AbortReq` in `managers/io_struct.py`.

## Shutdown

In [11]:
client.close()
server.should_exit = True